# Lecture 17: Validation And Resampling

This notebook compares holdout validation and cross-validation for housing price models.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
target = "price_k_eur"
X = housing[features]
y = housing[target]


In [ ]:
numeric = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]
categorical = ["district"]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ("num", "passthrough", numeric),
    ]
)

linear_model = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
linear_model.fit(X_train, y_train)
pred = linear_model.predict(X_test)
holdout_rmse = mean_squared_error(y_test, pred) ** 0.5
print(f"Holdout RMSE: {holdout_rmse:.2f}")


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    linear_model,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
print(f"Cross-validated RMSE: {-scores.mean():.2f}")
print(f"Fold-to-fold SD: {scores.std():.2f}")


In [ ]:
poly_preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ("poly_num", PolynomialFeatures(degree=2, include_bias=False), numeric),
    ]
)
poly_model = Pipeline(
    [
        ("preprocess", poly_preprocess),
        ("model", LinearRegression()),
    ]
)
poly_scores = cross_val_score(poly_model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
pd.DataFrame(
    {
        "model": ["linear", "polynomial_numeric"],
        "cv_rmse": [-scores.mean(), -poly_scores.mean()],
    }
)


## LLM Check

Ask an LLM whether the polynomial model is better. Require the answer to refer to the validation metric, uncertainty across folds, and model complexity.
